<a href="https://colab.research.google.com/github/mdmonsurali/Large-Language-Model-LLM-/blob/main/AI%20Agent/Smolagents/Build_Custom_document_agent_smolagent_ollama.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Hands-On Guide: Setting Up Ollama and Llama 3.1 on Google Colab or Locally
In this tutorial, we will go through a step-by-step guide to set up Ollama and Llama 3.1 for use on Google Colab or your local machine. We'll also explore how to utilize these tools with custom documents and enhance your agent with a retriever and web search.

# Step 1: Install and Setup Ollama

In [ ]:
!sudo apt update
!sudo apt install -y pciutils
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
import threading
import subprocess
import time

def run_ollama_serve():
    subprocess.Popen(["ollama", "serve"])

thread = threading.Thread(target=run_ollama_serve)
thread.start()
time.sleep(5)

In [ ]:
!ollama pull llama3.1

# Step 2: Install SmolAgent, other required library and Load the Model

In [ ]:
!pip install -q smolagents

In [ ]:
!pip install smolagents pandas langchain langchain-community sentence-transformers faiss-gpu --upgrade -q
!pip install PyMuPDF
!pip install rank_bm25 -q

In [ ]:
from smolagents import tool, LiteLLMModel

model = LiteLLMModel(model_id="ollama_chat/llama3.1", api_key="ollama")

# Step 3: Load and Process Custom Documents

In [ ]:
import os
import fitz

class Document:
    def __init__(self, page_content, metadata, doc_id):
        self.page_content = page_content
        self.metadata = metadata
        self.id = doc_id

def load_docs(directory):
    documents = []
    for idx, filename in enumerate(os.listdir(directory)):
        if filename.endswith(".pdf"):
            file_path = os.path.join(directory, filename)
            with fitz.open(file_path) as pdf_document:
                page_content = ""
                for page in pdf_document:
                    page_content += page.get_text("text")
                documents.append(Document(page_content, {"source": file_path}, doc_id=str(idx)))
    return documents

directory = "/content/sample_data/Doc"
docs = load_docs(directory)

In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

def split_docs(documents, chunk_size=500, chunk_overlap=20):
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
    docs = text_splitter.split_documents(documents)
    return docs

docs_processed = split_docs(docs)

print('Number of documents: ', len(docs))
print('Number of chunks: ', len(docs_processed))

# Step 4: Create a Retriever Tool

In [ ]:
from smolagents import Tool
from langchain.retrievers import BM25Retriever

class RetrieverTool(Tool):
    name = "retriever"
    description = "Uses semantic search to retrieve relevant parts of the documents."
    inputs = {
        "query": {
            "type": "string",
            "description": "The query to perform. Use affirmative sentences rather than questions.",
        }
    }
    output_type = "string"

    def __init__(self, docs, **kwargs):
        super().__init__(**kwargs)
        self.retriever = BM25Retriever.from_documents(docs, k=10)

    def forward(self, query: str) -> str:
        assert isinstance(query, str), "Your search query must be a string"
        docs = self.retriever.invoke(query)
        return "\nRetrieved documents:\n" + "".join([
            f"\n\n===== Document {str(i)} =====\n" + doc.page_content
            for i, doc in enumerate(docs)
        ])

retriever_tool = RetrieverTool(docs_processed)

# Step 5: Build and Run the Agent

In [ ]:
from smolagents import CodeAgent

agent = CodeAgent(
    tools=[retriever_tool], model=model, max_iterations=4, verbose=True
)

agent_output = agent.run("What happens after an over is completed in a cricket match?")

print("Final output:")
print(agent_output)

In [ ]:
from smolagents import CodeAgent, DuckDuckGoSearchTool

agent = CodeAgent(
    tools=[retriever_tool, DuckDuckGoSearchTool], model=model, max_iterations=4, verbose=True
)

agent.run("What happens when a footballer receives a red card?")

In [ ]:
import nbformat
with open('/content/Build_Custom_document_agent_smolagent_ollama.ipynb') as f:
    nbformat.read(f, as_version=4)


In [ ]:
import json
import nbformat

file_path = '/content/Build_Custom_document_agent_smolagent_ollama.ipynb'
repaired_file_path = 'repaired_notebook.ipynb'

# Attempt to read and repair
try:
    with open(file_path, 'r') as f:
        data = json.load(f)  # Load JSON data

    # Validate as a Jupyter Notebook
    nbformat.validate(data)

    # Save a valid file
    with open(repaired_file_path, 'w') as f:
        json.dump(data, f, indent=4)
    print(f"Notebook repaired and saved as {repaired_file_path}")
except Exception as e:
    print(f"Error: {e}")
